# Azure Cosmos DB No SQL

本笔记本展示了如何利用这个集成的[向量数据库](https://learn.microsoft.com/en-us/azure/cosmos-db/vector-database)在集合中存储文档、创建索引并执行向量搜索查询，使用近似最近邻算法，如COS（余弦距离）、L2（欧氏距离）和IP（内积），以查找与查询向量接近的文档。

Azure Cosmos DB 是驱动 OpenAI 的 ChatGPT 服务的数据库。它提供个位数毫秒级的响应时间、自动即时可扩展性，以及任何规模下的速度保证。

Azure Cosmos DB for NoSQL 目前预览版提供向量索引和搜索功能。此功能旨在处理高维向量，从而在任何规模下实现高效准确的向量搜索。您现在可以将向量直接存储在文档中，与您的数据并列。这意味着数据库中的每个文档不仅可以包含传统的无模式数据，还可以包含高维向量作为文档的其他属性。这种数据和向量的共置允许高效的索引和搜索，因为向量与它们所代表的数据存储在相同的逻辑单元中。这简化了数据管理、AI 应用程序架构以及基于向量的操作的效率。

更多详情请参阅：
- [向量搜索](https://learn.microsoft.com/en-us/azure/cosmos-db/nosql/vector-search)
- [全文搜索](https://learn.microsoft.com/en-us/azure/cosmos-db/gen-ai/full-text-search)
- [混合搜索](https://learn.microsoft.com/en-us/azure/cosmos-db/gen-ai/hybrid-search)

[注册](https://azure.microsoft.com/en-us/free/)即可获得终身免费访问权限，立即开始。

In [1]:
%pip install --upgrade --quiet azure-cosmos langchain-openai langchain-community

Note: you may need to restart the kernel to use updated packages.


In [1]:
OPENAI_API_KEY = "YOUR_KEY"
OPENAI_API_TYPE = "azure"
OPENAI_API_VERSION = "2023-05-15"
OPENAI_API_BASE = "YOUR_ENDPOINT"
OPENAI_EMBEDDINGS_MODEL_NAME = "text-embedding-ada-002"
OPENAI_EMBEDDINGS_MODEL_DEPLOYMENT = "text-embedding-ada-002"

## 插入数据

In [2]:
from langchain_community.document_loaders import PyPDFLoader

# Load the PDF
loader = PyPDFLoader("https://arxiv.org/pdf/2303.08774.pdf")
data = loader.load()

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
docs = text_splitter.split_documents(data)

In [4]:
print(docs[0])

page_content='GPT-4 Technical Report
OpenAI∗
Abstract
We report the development of GPT-4, a large-scale, multimodal model which can
accept image and text inputs and produce text outputs. While less capable than
humans in many real-world scenarios, GPT-4 exhibits human-level performance
on various professional and academic benchmarks, including passing a simulated
bar exam with a score around the top 10% of test takers. GPT-4 is a Transformer-
based model pre-trained to predict the next token in a document. The post-training
alignment process results in improved performance on measures of factuality and
adherence to desired behavior. A core component of this project was developing
infrastructure and optimization methods that behave predictably across a wide
range of scales. This allowed us to accurately predict some aspects of GPT-4’s
performance based on models trained with no more than 1/1,000th the compute of
GPT-4.
1 Introduction' metadata={'source': 'https://arxiv.org/pdf/2303.0877

## 创建 Azure Cosmos DB NoSQL 向量搜索

Azure Cosmos DB for NoSQL 现已支持向量搜索。

向量搜索是一种允许您使用向量嵌入来搜索非结构化数据的技术。向量嵌入是数据的数值表示，可以用来表示文本、图像、音频和视频等。

您可以使用向量搜索来构建各种应用程序，包括：

* 语义搜索
* 推荐系统
* 异常检测
* 图像搜索

Azure Cosmos DB for NoSQL 中的向量搜索功能基于 Azure OpenAI 服务。Azure OpenAI 服务提供了一系列预训练的语言模型，可以用来生成向量嵌入。

要使用 Azure Cosmos DB for NoSQL 中的向量搜索功能，您需要执行以下步骤：

1. **创建 Azure Cosmos DB for NoSQL 帐户。** 如果您还没有 Azure Cosmos DB for NoSQL 帐户，可以免费创建一个。

2. **创建容器。** 在您的 Azure Cosmos DB for NoSQL 帐户中创建一个容器。

3. **启用向量搜索。** 在您的容器中启用向量搜索。

4. **创建向量索引。** 在您的容器中创建一个向量索引。向量索引是用来加速向量搜索的。

5. **将数据插入容器。** 将您的数据插入容器。数据应包含向量嵌入。

6. **执行向量搜索。** 使用向量搜索来搜索您的数据。

以下是使用 Azure Cosmos DB for NoSQL 创建向量搜索的详细步骤：

### 1. 创建 Azure Cosmos DB for NoSQL 帐户

如果您还没有 Azure Cosmos DB for NoSQL 帐户，可以免费创建一个。有关如何创建帐户的更多信息，请访问 [Azure Cosmos DB 文档](https://docs.microsoft.com/azure/cosmos-db/nosql/quickstart-portal)。

### 2. 创建容器

在您的 Azure Cosmos DB for NoSQL 帐户中创建一个容器。有关如何创建容器的更多信息，请访问 [Azure Cosmos DB 文档](https://docs.microsoft.com/azure/cosmos-db/nosql/quickstart-portal)。

### 3. 启用向量搜索

在您的容器中启用向量搜索。您可以通过 Azure 门户、Azure CLI 或 Azure Cosmos DB SDK 来完成此操作。

**使用 Azure 门户启用向量搜索：**

1. 导航到您的 Azure Cosmos DB for NoSQL 帐户。
2. 在左侧菜单中，选择“数据资源管理器”。
3. 选择您的容器。
4. 在容器设置中，选择“功能”。
5. 启用“向量搜索”。

**使用 Azure CLI 启用向量搜索：**

```bash
az cosmosdb sql container update --account-name <your-account-name> --resource-group <your-resource-group> --name <your-container-name> --enable-vector-search true
```

### 4. 创建向量索引

在您的容器中创建一个向量索引。向量索引是用来加速向量搜索的。您可以通过 Azure 门户、Azure CLI 或 Azure Cosmos DB SDK 来完成此操作。

**使用 Azure 门户创建向量索引：**

1. 导航到您的 Azure Cosmos DB for NoSQL 帐户。
2. 在左侧菜单中，选择“数据资源管理器”。
3. 选择您的容器。
4. 在容器设置中，选择“索引”。
5. 选择“添加索引”。
6. 选择“向量索引”。
7. 指定向量字段的名称和向量嵌入的维度。
8. 选择索引类型。最常见的类型是 `hnsw` (Hierarchical Navigable Small Worlds)。
9. 选择距离度量。最常见的度量是 `cosine` 或 `euclidean`。
10. 单击“保存”。

**使用 Azure CLI 创建向量索引：**

```bash
az cosmosdb sql container index create --account-name <your-account-name> --resource-group <your-resource-group> --container-name <your-container-name> --name <your-index-name> --vector-index '{ "paths": [ { "path": "/vector", "vectorIndex": { "type": "hnsw", "dimensions": 1536, "distanceFunction": "cosine" } } ] }'
```

### 5. 将数据插入容器

将您的数据插入容器。数据应包含向量嵌入。您可以使用 Azure Cosmos DB SDK 或 REST API 来完成此操作。

以下是使用 Python SDK 将数据插入容器的示例：

```python
from azure.cosmos import CosmosClient

# Replace with your Cosmos DB endpoint and key
endpoint = "https://<your-account-name>.documents.azure.com:443/"
key = "<your-key>"

client = CosmosClient(endpoint, key)

# Replace with your database and container names
database_name = "mydatabase"
container_name = "mycontainer"

database = client.get_database_client(database_name)
container = database.get_container_client(container_name)

# Sample data with a vector embedding
item = {
    "id": "item1",
    "content": "This is a sample document.",
    "vector": [0.1, 0.2, 0.3, ..., 0.9] # Replace with your actual vector embedding
}

container.create_item(item)
```

### 6. 执行向量搜索

使用向量搜索来搜索您的数据。您可以使用 Azure Cosmos DB SDK 或 REST API 来完成此操作。

以下是使用 Python SDK 执行向量搜索的示例：

```python
from azure.cosmos import CosmosClient

# Replace with your Cosmos DB endpoint and key
endpoint = "https://<your-account-name>.documents.azure.com:443/"
key = "<your-key>"

client = CosmosClient(endpoint, key)

# Replace with your database and container names
database_name = "mydatabase"
container_name = "mycontainer"

database = client.get_database_client(database_name)
container = database.get_container_client(container_name)

# Sample query vector
query_vector = [0.1, 0.2, 0.3, ..., 0.9] # Replace with your actual query vector

# Perform a vector search
results = container.query_items(
    query="SELECT TOP 5 * FROM c ORDER BY VectorDistance(c.vector, @queryVector) ASC",
    parameters=[
        {"name": "@queryVector", "value": query_vector}
    ]
)

for item in results:
    print(item)

In [5]:
indexing_policy = {
    "indexingMode": "consistent",
    "includedPaths": [{"path": "/*"}],
    "excludedPaths": [{"path": '/"_etag"/?'}],
    "vectorIndexes": [{"path": "/embedding", "type": "diskANN"}],
    "fullTextIndexes": [{"path": "/text"}],
}

vector_embedding_policy = {
    "vectorEmbeddings": [
        {
            "path": "/embedding",
            "dataType": "float32",
            "distanceFunction": "cosine",
            "dimensions": 1536,
        }
    ]
}

full_text_policy = {
    "defaultLanguage": "en-US",
    "fullTextPaths": [{"path": "/text", "language": "en-US"}],
}

In [10]:
from azure.cosmos import CosmosClient, PartitionKey
from langchain_community.vectorstores.azure_cosmos_db_no_sql import (
    AzureCosmosDBNoSqlVectorSearch,
)
from langchain_openai import OpenAIEmbeddings

HOST = "AZURE_COSMOS_DB_ENDPOINT"
KEY = "AZURE_COSMOS_DB_KEY"

cosmos_client = CosmosClient(HOST, KEY)
database_name = "langchain_python_db"
container_name = "langchain_python_container"
partition_key = PartitionKey(path="/id")
cosmos_container_properties = {"partition_key": partition_key}

openai_embeddings = OpenAIEmbeddings(
    deployment="smart-agent-embedding-ada",
    model="text-embedding-ada-002",
    chunk_size=1,
    openai_api_key="OPENAI_API_KEY",
)

# insert the documents in AzureCosmosDBNoSql with their embedding
vector_search = AzureCosmosDBNoSqlVectorSearch.from_documents(
    documents=docs,
    embedding=openai_embeddings,
    cosmos_client=cosmos_client,
    database_name=database_name,
    container_name=container_name,
    vector_embedding_policy=vector_embedding_policy,
    full_text_policy=full_text_policy,
    indexing_policy=indexing_policy,
    cosmos_container_properties=cosmos_container_properties,
    cosmos_database_properties={},
    full_text_search_enabled=True,
)

## 向量搜索

Vector search is a method for searching for information that is similar to a given query, based on the semantic meaning of the data. It is particularly useful for unstructured data such as text, images, and audio.

Vector search works by converting data into numerical representations called **vectors** or **embeddings**. These vectors capture the semantic meaning of the data. For example, the vectors for "dog" and "puppy" would be closer together in the vector space than the vectors for "dog" and "car".

Once the data is converted into vectors, a vector database can be used to store and index these vectors. When a query is made, it is also converted into a vector. The vector database then searches for vectors that are closest to the query vector, using algorithms like **Approximate Nearest Neighbor (ANN)**. The results are then returned as the most semantically similar pieces of information to the query.

**Key Concepts:**

*   **Vectors/Embeddings:** Numerical representations of data that capture semantic meaning.
*   **Vector Database:** A specialized database designed to store, index, and search vector data efficiently.
*   **Approximate Nearest Neighbor (ANN):** Algorithms used to find approximate nearest neighbors in a high-dimensional space, which is much faster than exact nearest neighbor search.

**Use Cases:**

*   **Semantic Search:** Finding information based on meaning rather than keywords.
*   **Recommendation Systems:** Suggesting similar items (e.g., products, movies, music) to users.
*   **Image and Audio Search:** Finding similar images or audio clips.
*   **Natural Language Processing (NLP):** Tasks like question answering, text summarization, and sentiment analysis.

Vector search is a powerful technique that enables more intelligent and context-aware information retrieval, especially for complex and unstructured data.

In [22]:
# Perform a similarity search between the embedding of the query and the embeddings of the documents
import json

query = "What were the compute requirements for training GPT 4"
results = vector_search.similarity_search(query)

print(results[0].page_content)

performance based on models trained with no more than 1/1,000th the compute of
GPT-4.
1 Introduction
This technical report presents GPT-4, a large multimodal model capable of processing image and
text inputs and producing text outputs. Such models are an important area of study as they have the
potential to be used in a wide range of applications, such as dialogue systems, text summarization,
and machine translation. As such, they have been the subject of substantial interest and progress in
recent years [1–34].
One of the main goals of developing such models is to improve their ability to understand and generate
natural language text, particularly in more complex and nuanced scenarios. To test its capabilities
in such scenarios, GPT-4 was evaluated on a variety of exams originally designed for humans. In
these evaluations it performs quite well and often outscores the vast majority of human test takers.


## 带分数的向量搜索

Vector search, also known as similarity search, is a method of retrieving documents that are similar to a given query. This is achieved by representing both the documents and the query as vectors in a high-dimensional space. The similarity between vectors is then measured using a distance metric, such as cosine similarity or Euclidean distance.

向量搜索，也称为相似性搜索，是一种检索与给定查询相似的文档的方法。这通过将文档和查询都表示为高维空间中的向量来实现。然后，使用距离度量（如余弦相似度或欧几里得距离）来衡量向量之间的相似性。

In many applications, it is useful to not only retrieve similar documents but also to rank them based on their similarity score. This allows users to prioritize the most relevant results.

在许多应用中，不仅检索相似文档而且根据其相似性分数对它们进行排名非常有用。这允许用户优先处理最相关的结果。

### How it works

### 工作原理

1.  **Vectorization**: Documents and the query are converted into numerical vectors. This is typically done using embedding models, such as those based on deep learning.

    1.  **向量化**: 文档和查询被转换为数值向量。这通常使用嵌入模型完成，例如基于深度学习的模型。

2.  **Indexing**: The document vectors are stored in an efficient data structure called a vector index. This index allows for fast searching of similar vectors.

    2.  **索引**: 文档向量存储在一种称为向量索引的高效数据结构中。此索引允许快速搜索相似向量。

3.  **Querying**: The query vector is used to search the index for the nearest neighbors (i.e., the most similar document vectors).

    3.  **查询**: 查询向量用于在索引中搜索最近邻（即最相似的文档向量）。

4.  **Scoring and Ranking**: The similarity between the query vector and each retrieved document vector is calculated using a chosen distance metric. The documents are then ranked in descending order of their scores.

    4.  **评分和排名**: 使用选定的距离度量来计算查询向量与每个检索到的文档向量之间的相似性。然后，文档按分数降序排名。

### Use cases

### 用例

*   **Recommendation Systems**: Recommending products, articles, or music based on user preferences or item similarity.
*   **Image Search**: Finding images that are visually similar to a query image.
*   **Natural Language Processing**: Semantic search, question answering, and document summarization.
*   **Anomaly Detection**: Identifying unusual patterns or outliers in data.

*   **推荐系统**: 根据用户偏好或物品相似性推荐产品、文章或音乐。
*   **图像搜索**: 查找在视觉上与查询图像相似的图像。
*   **自然语言处理**: 语义搜索、问答和文档摘要。
*   **异常检测**: 识别数据中的异常模式或离群值。

### Example

### 示例

Let's say we have the following documents:

假设我们有以下文档：

*   **Document 1**: "The quick brown fox jumps over the lazy dog."
*   **Document 2**: "A fast, dark-colored fox leaps above a sleepy canine."
*   **Document 3**: "The weather today is sunny and warm."

*   **文档 1**: "敏捷的棕色狐狸跳过懒狗。"
*   **文档 2**: "一只快速的深色狐狸跃过一只昏昏欲睡的犬科动物。"
*   **文档 3**: "今天天气晴朗温暖。"

And our query is:

我们的查询是：

*   **Query**: "A speedy fox leaps."

*   **查询**: "一只快速的狐狸跳跃。"

After vectorization and similarity calculation (using cosine similarity, for example), we might get the following scores:

向量化和相似性计算（例如，使用余弦相似度）后，我们可能会得到以下分数：

*   **Document 1**: Score: 0.75
*   **Document 2**: Score: 0.92
*   **Document 3**: Score: 0.15

*   **文档 1**: 分数: 0.75
*   **文档 2**: 分数: 0.92
*   **文档 3**: 分数: 0.15

Based on these scores, the ranked results would be:

根据这些分数，排名结果将是：

1.  **Document 2** (Score: 0.92)
2.  **Document 1** (Score: 0.75)
3.  **Document 3** (Score: 0.15)

1.  **文档 2** (分数: 0.92)
2.  **文档 1** (分数: 0.75)
3.  **文档 3** (分数: 0.15)

This demonstrates how vector search with scores provides a ranked list of relevant results, allowing users to easily identify the most similar documents.

这展示了带分数的向量搜索如何提供相关结果的排名列表，使用户能够轻松识别最相似的文档。

In [44]:
query = "What were the compute requirements for training GPT 4"

results = vector_search.similarity_search_with_score(
    query=query,
    k=5,
)

# Display results
for i in range(0, len(results)):
    print(f"Result {i+1}: ", results[i][0].json())
    print(f"Score {i+1}: ", results[i][1])
    print("\n")

Result 1:  {"id":null,"metadata":{"source":"https://arxiv.org/pdf/2303.08774.pdf","page":0,"id":"9d59c3ed-deac-48cb-9382-a8ab079334e5"},"page_content":"performance based on models trained with no more than 1/1,000th the compute of\nGPT-4.\n1 Introduction\nThis technical report presents GPT-4, a large multimodal model capable of processing image and\ntext inputs and producing text outputs. Such models are an important area of study as they have the\npotential to be used in a wide range of applications, such as dialogue systems, text summarization,\nand machine translation. As such, they have been the subject of substantial interest and progress in\nrecent years [1–34].\nOne of the main goals of developing such models is to improve their ability to understand and generate\nnatural language text, particularly in more complex and nuanced scenarios. To test its capabilities\nin such scenarios, GPT-4 was evaluated on a variety of exams originally designed for humans. In\nthese evaluations it

## 带过滤条件的向量搜索

Vector search allows you to find items that are similar to a given query vector. However, in many real-world applications, you also need to filter the search results based on certain criteria. For example, you might want to find similar products but only those that are currently in stock or belong to a specific category.

This guide explains how to perform vector search with filtering in [Your Database System Name].

### Prerequisites

*   A database with vector data indexed.
*   Basic understanding of vector search and filtering concepts.

### Performing Vector Search with Filtering

To perform vector search with filtering, you typically combine a vector similarity search with a metadata filter. The exact syntax and methods may vary depending on the database system you are using.

Here's a general approach:

1.  **Define your query vector:** This is the vector representing your search query.
2.  **Define your metadata filter:** This specifies the criteria you want to filter your results by. This could be a JSON object, a set of key-value pairs, or a specific query language.
3.  **Execute the search:** Use your database's specific API or query language to perform a search that includes both the query vector and the metadata filter.

#### Example (Conceptual)

Let's assume you have a collection of products, each with a `vector` (for similarity search) and `metadata` (containing `category` and `in_stock` fields).

**Query:** Find products similar to a given `query_vector`, where the `category` is "electronics" and `in_stock` is `true`.

**Conceptual Query:**

```sql
SELECT *
FROM products
WHERE
  vector_similarity(vector, :query_vector) > :similarity_threshold
  AND metadata.category = 'electronics'
  AND metadata.in_stock = true
ORDER BY vector_similarity(vector, :query_vector) DESC
LIMIT 10;
```

**Explanation:**

*   `vector_similarity(vector, :query_vector)`: This function calculates the similarity between the product's vector and your query vector.
*   `:similarity_threshold`: A parameter to define the minimum similarity score.
*   `metadata.category = 'electronics'`: This is the metadata filter for the category.
*   `metadata.in_stock = true`: This is the metadata filter for stock availability.
*   `ORDER BY ... DESC`: Sorts the results by similarity in descending order.
*   `LIMIT 10`: Limits the number of results returned.

**Note:** The actual syntax for `vector_similarity`, metadata access, and filtering will depend on your specific database system (e.g., PostgreSQL with `pgvector`, Elasticsearch, Pinecone, Weaviate, etc.).

### Specific Database Implementations

Please refer to the documentation of your specific database system for detailed instructions and examples:

*   **[Your Database System Name] Documentation on Vector Search with Filtering:** [Link to relevant documentation]

By combining vector similarity search with metadata filtering, you can build more powerful and relevant search applications.## 带过滤条件的向量搜索

向量搜索允许您查找与给定查询向量相似的项目。然而，在许多实际应用中，您还需要根据特定条件过滤搜索结果。例如，您可能希望查找相似的产品，但只查找当前有货或属于特定类别的产品。

本指南将介绍如何在 [您的数据库系统名称] 中执行带过滤条件的向量搜索。

### 先决条件

*   一个已建立向量索引的数据库。
*   对向量搜索和过滤概念的基本理解。

### 执行带过滤条件的向量搜索

要执行带过滤条件的向量搜索，通常需要将向量相似性搜索与元数据过滤器结合起来。确切的语法和方法可能因您使用的数据库系统而异。

以下是一种通用方法：

1.  **定义查询向量：** 这是代表您的搜索查询的向量。
2.  **定义元数据过滤器：** 这指定了您要通过其过滤结果的标准。这可以是一个 JSON 对象、一组键值对或特定的查询语言。
3.  **执行搜索：** 使用您的数据库的特定 API 或查询语言执行包含查询向量和元数据过滤器的搜索。

#### 示例（概念性）

假设您有一个产品集合，每个产品都有一个 `vector`（用于相似性搜索）和 `metadata`（包含 `category` 和 `in_stock` 字段）。

**查询：** 查找与给定 `query_vector` 相似的产品，其中 `category` 为“electronics”，`in_stock` 为 `true`。

**概念性查询：**

```sql
SELECT *
FROM products
WHERE
  vector_similarity(vector, :query_vector) > :similarity_threshold
  AND metadata.category = 'electronics'
  AND metadata.in_stock = true
ORDER BY vector_similarity(vector, :query_vector) DESC
LIMIT 10;
```

**说明：**

*   `vector_similarity(vector, :query_vector)`：此函数计算产品向量与您的查询向量之间的相似性。
*   `:similarity_threshold`：定义最小相似性分数的参数。
*   `metadata.category = 'electronics'`：这是类别的元数据过滤器。
*   `metadata.in_stock = true`：这是库存可用性的元数据过滤器。
*   `ORDER BY ... DESC`：按相似性降序对结果进行排序。
*   `LIMIT 10`：限制返回的结果数量。

**注意：** `vector_similarity`、元数据访问和过滤器的实际语法将取决于您的特定数据库系统（例如，PostgreSQL 与 `pgvector`、Elasticsearch、Pinecone、Weaviate 等）。

### 特定数据库实现

请参阅您特定数据库系统的文档，了解详细说明和示例：

*   **[您的数据库系统名称] 关于带过滤条件的向量搜索的文档：** [相关文档链接]

通过将向量相似性搜索与元数据过滤相结合，您可以构建更强大、更相关的搜索应用程序。

In [50]:
query = "What were the compute requirements for training GPT 4"

pre_filter = {
    "conditions": [
        {"property": "metadata.page", "operator": "$eq", "value": 0},
    ],
}

results = vector_search.similarity_search_with_score(
    query=query,
    k=5,
    pre_filter=pre_filter,
)

# Display results
for i in range(0, len(results)):
    print(f"Result {i+1}: ", results[i][0].json())
    print(f"Score {i+1}: ", results[i][1])
    print("\n")

Result 1:  {"id":null,"metadata":{"source":"https://arxiv.org/pdf/2303.08774.pdf","page":0,"id":"9d59c3ed-deac-48cb-9382-a8ab079334e5"},"page_content":"performance based on models trained with no more than 1/1,000th the compute of\nGPT-4.\n1 Introduction\nThis technical report presents GPT-4, a large multimodal model capable of processing image and\ntext inputs and producing text outputs. Such models are an important area of study as they have the\npotential to be used in a wide range of applications, such as dialogue systems, text summarization,\nand machine translation. As such, they have been the subject of substantial interest and progress in\nrecent years [1–34].\nOne of the main goals of developing such models is to improve their ability to understand and generate\nnatural language text, particularly in more complex and nuanced scenarios. To test its capabilities\nin such scenarios, GPT-4 was evaluated on a variety of exams originally designed for humans. In\nthese evaluations it

## 全文搜索

Full-text search (FTS) is a search technique that finds documents that match a query consisting of several words. It is different from **keyword search**, which only finds documents that contain the exact keyword.

全文搜索（FTS）是一种搜索技术，用于查找与由多个单词组成的查询匹配的文档。这与**关键词搜索**不同，后者仅查找包含精确关键词的文档。

For example, if you search for "quick brown fox", a full-text search might return documents that contain "the quick brown fox jumps over the lazy dog", while a keyword search would only return documents that contain the exact phrase "quick brown fox".

例如，如果您搜索“quick brown fox”，全文搜索可能会返回包含“the quick brown fox jumps over the lazy dog”的文档，而关键词搜索将仅返回包含精确短语“quick brown fox”的文档。

Full-text search is often used in applications such as:

全文搜索常用于以下类型的应用程序：

*   **Databases:** Many databases offer full-text search capabilities, allowing users to search for text within database records.
*   **Document management systems:** These systems use full-text search to help users find specific documents within a large collection.
*   **Web search engines:** Search engines like Google and Bing use sophisticated full-text search algorithms to index and retrieve web pages.

*   **数据库：**许多数据库提供全文搜索功能，允许用户在数据库记录中搜索文本。
*   **文档管理系统：**这些系统使用全文搜索来帮助用户在大量文档中查找特定文档。
*   **网络搜索引擎：**像Google和Bing这样的搜索引擎使用复杂的全文搜索算法来索引和检索网页。

There are many different algorithms and techniques used for full-text search, but they generally involve the following steps:

有许多不同的算法和技术用于全文搜索，但它们通常涉及以下步骤：

1.  **Indexing:** The system creates an index of the words in the documents. This index is typically a data structure that maps words to the documents in which they appear.
2.  **Querying:** When a user submits a query, the system searches the index for documents that match the query.
3.  **Ranking:** The system ranks the matching documents based on their relevance to the query. This ranking can be based on various factors, such as the number of times the query words appear in the document, the proximity of the query words to each other, and the importance of the documents themselves.

1.  **索引：**系统会创建文档中单词的索引。该索引通常是一种数据结构，将单词映射到它们出现的文档。
2.  **查询：**当用户提交查询时，系统会在索引中搜索与查询匹配的文档。
3.  **排名：**系统根据匹配文档与查询的相关性对其进行排名。此排名可以基于各种因素，例如查询词在文档中出现的次数、查询词之间的接近程度以及文档本身的重要性。

Some common full-text search techniques include:

一些常见的全文搜索技术包括：

*   **Inverted index:** This is the most common technique for full-text search. It is a data structure that maps words to the documents in which they appear.
*   **Suffix array:** This is a data structure that stores all suffixes of a string. It can be used for full-text search by searching for the query string in the suffix array.
*   **N-gram:** This technique breaks down text into overlapping sequences of characters or words. It can be used for full-text search by searching for matching n-grams in the documents.

*   **倒排索引：**这是全文搜索最常用的技术。它是一种将单词映射到它们出现的文档的数据结构。
*   **后缀数组：**这是一种存储字符串所有后缀的数据结构。通过在后缀数组中搜索查询字符串，可用于全文搜索。
*   **N-gram：**该技术将文本分解为重叠的字符或单词序列。通过在文档中搜索匹配的n-gram，可用于全文搜索。

Full-text search is a powerful tool that can be used to improve the usability of applications by making it easier for users to find the information they need.

全文搜索是一个强大的工具，可以通过使用户更容易找到所需信息来提高应用程序的可用性。

In [42]:
from langchain_community.vectorstores.azure_cosmos_db_no_sql import CosmosDBQueryType

query = "What were the compute requirements for training GPT 4"
pre_filter = {
    "conditions": [
        {
            "property": "text",
            "operator": "$full_text_contains_any",
            "value": "What were the compute requirements for training GPT 4",
        },
    ],
}
results = vector_search.similarity_search_with_score(
    query=query,
    k=5,
    query_type=CosmosDBQueryType.FULL_TEXT_SEARCH,
    pre_filter=pre_filter,
)

# Display results
for i in range(0, len(results)):
    print(f"Result {i+1}: ", results[i][0].json())
    print("\n")

Result 1:  {"id":null,"metadata":{"source":"https://arxiv.org/pdf/2303.08774.pdf","page":0,"id":"cddfb7ac-d953-46f4-8a48-76655f116bcf"},"page_content":"GPT-4 Technical Report\nOpenAI∗\nAbstract\nWe report the development of GPT-4, a large-scale, multimodal model which can\naccept image and text inputs and produce text outputs. While less capable than\nhumans in many real-world scenarios, GPT-4 exhibits human-level performance\non various professional and academic benchmarks, including passing a simulated\nbar exam with a score around the top 10% of test takers. GPT-4 is a Transformer-\nbased model pre-trained to predict the next token in a document. The post-training\nalignment process results in improved performance on measures of factuality and\nadherence to desired behavior. A core component of this project was developing\ninfrastructure and optimization methods that behave predictably across a wide\nrange of scales. This allowed us to accurately predict some aspects of GPT-4’s\nper

## 全文搜索 BM25 排序

BM25 is a ranking function used by search engines to estimate the relevance of documents to a given search query. It is a **probabilistic model** based on the **probabilistic retrieval framework**.

BM25 is a **family of scoring functions**, with BM25F being the most commonly used variant. It is a **generalization of the binary independence model (BIM)**.

BM25 is **parameterized by k1 and b**.

- **k1** is a parameter that controls the **term frequency saturation**. A higher k1 value means that the term frequency has a **lesser impact** on the score.
- **b** is a parameter that controls the **document length normalization**. A higher b value means that the document length has a **greater impact** on the score.

The BM25 scoring function is defined as:

```
score(D, q) = Σ [ log( (r + 0.5) / (R - r + 0.5) ) * ( (k1 + 1) * tf / (tf + k1 * (1 - b + b * |D| / avgdl)) ) * ( (k2 + 1) * qtf / (qtf + k2) ) ]
```

Where:

- **D** is a document
- **q** is the search query
- **r** is the number of documents that contain the term
- **R** is the total number of documents
- **tf** is the term frequency of the term in the document
- **|D|** is the length of the document
- **avgdl** is the average document length
- **k1** and **b** are parameters
- **k2** is a parameter that controls the **query term frequency saturation**. A higher k2 value means that the query term frequency has a **lesser impact** on the score.
- **qtf** is the term frequency of the term in the query

BM25 is a **state-of-the-art ranking function** that is widely used in search engines. It is a **robust and effective** ranking function that can be used to improve the relevance of search results.

Here are some of the advantages of using BM25:

- **It is a probabilistic model**, which means that it is based on a solid theoretical foundation.
- **It is a generalization of the binary independence model**, which means that it is more powerful than the BIM.
- **It is parameterized**, which means that it can be tuned to the specific needs of a search engine.
- **It is robust and effective**, which means that it can be used to improve the relevance of search results.

Here are some of the disadvantages of using BM25:

- **It can be computationally expensive**, especially for large collections of documents.
- **It can be sensitive to the choice of parameters**, which can be difficult to tune.
- **It does not take into account the semantic meaning of words**, which can lead to irrelevant results.

Despite its disadvantages, BM25 is still a **widely used and effective ranking function**. It is a good choice for search engines that need a **robust and effective** ranking function.

In [47]:
query = "What were the compute requirements for training GPT 4"

results = vector_search.similarity_search_with_score(
    query=query,
    k=5,
    query_type=CosmosDBQueryType.FULL_TEXT_RANK,
)

# Display results
for i in range(0, len(results)):
    print(f"Result {i+1}: ", results[i][0].json())
    print("\n")

Result 1:  {"id":null,"metadata":{"source":"https://arxiv.org/pdf/2303.08774.pdf","page":2,"id":"f2746fd3-bbcb-4197-b2d5-ee7b355b6009"},"page_content":"the HumanEval dataset. A power law fit to the smaller models (excluding GPT-4) is shown as the dotted\nline; this fit accurately predicts GPT-4’s performance. The x-axis is training compute normalized so that\nGPT-4 is 1.\n3","type":"Document"}


Result 2:  {"id":null,"metadata":{"source":"https://arxiv.org/pdf/2303.08774.pdf","page":1,"id":"20153a6c-7c2c-4328-9b0e-e3502d7ac4dd"},"page_content":"safety considerations above against the scientific value of further transparency.\n3 Predictable Scaling\nA large focus of the GPT-4 project was building a deep learning stack that scales predictably. The\nprimary reason is that for very large training runs like GPT-4, it is not feasible to do extensive\nmodel-specific tuning. To address this, we developed infrastructure and optimization methods that\nhave very predictable behavior across multip

## 混合搜索

Hybrid search combines the strengths of both keyword-based search and semantic search.

Keyword search is good at matching exact terms and phrases. For example, if you search for "apple pie recipe," a keyword search will look for documents containing those exact words. This is useful for finding specific information when you know the exact terms to use.

Semantic search, on the other hand, understands the meaning and context of your query. It can find documents that are conceptually related to your search, even if they don't contain the exact keywords. For example, if you search for "dessert with apples and cinnamon," a semantic search might return results for "apple pie," "apple crumble," or "baked apples," because it understands the underlying meaning of your query.

Hybrid search aims to leverage the benefits of both approaches. It typically works by:

1.  **Performing a keyword search:** This retrieves documents that contain the exact terms in your query.
2.  **Performing a semantic search:** This retrieves documents that are semantically similar to your query.
3.  **Combining and ranking the results:** The results from both searches are then combined, and a ranking algorithm is used to prioritize the most relevant documents. This algorithm might consider factors such as the number of keyword matches, the semantic similarity score, and other relevance signals.

The advantage of hybrid search is that it can provide more comprehensive and accurate results than either keyword or semantic search alone. It can help you find information even when you don't know the exact keywords, while still ensuring that specific, relevant results are surfaced.

**Use cases for hybrid search include:**

*   **E-commerce:** Helping customers find products even if they use varied terminology or descriptions.
*   **Document retrieval:** Finding relevant internal documents or knowledge base articles.
*   **Customer support:** Identifying solutions to customer issues based on natural language descriptions.
*   **Content discovery:** Recommending articles, videos, or other content that aligns with user interests.

By combining keyword and semantic matching, hybrid search offers a more robust and user-friendly search experience.

---

混合搜索结合了基于关键词的搜索和语义搜索的优点。

关键词搜索擅长匹配精确的词语和短语。例如，如果您搜索“apple pie recipe”（苹果派食谱），关键词搜索将查找包含这些确切词语的文档。这对于在知道确切用词时查找特定信息非常有用。

另一方面，语义搜索能够理解查询的含义和上下文。即使文档不包含确切的关键词，它也能找到与您的查询概念上相关的文档。例如，如果您搜索“dessert with apples and cinnamon”（带有苹果和肉桂的甜点），语义搜索可能会返回“apple pie”（苹果派）、“apple crumble”（苹果酥）或“baked apples”（烤苹果）的结果，因为它理解您查询的潜在含义。

混合搜索旨在利用这两种方法的优势。它通常通过以下方式工作：

1.  **执行关键词搜索：** 这会检索包含查询中确切词语的文档。
2.  **执行语义搜索：** 这会检索与您的查询在语义上相似的文档。
3.  **合并和排名结果：** 然后合并来自两种搜索的结果，并使用排名算法来优先排序最相关的文档。该算法可能会考虑诸如关键词匹配数量、语义相似度得分以及其他相关性信号等因素。

混合搜索的优势在于，与单独使用关键词搜索或语义搜索相比，它可以提供更全面、更准确的结果。它可以在您不知道确切关键词的情况下帮助您找到信息，同时仍确保能够呈现特定、相关的结果。

**混合搜索的用例包括：**

*   **电子商务：** 帮助客户找到产品，即使他们使用的术语或描述各不相同。
*   **文档检索：** 查找相关的内部文档或知识库文章。
*   **客户支持：** 根据自然语言描述识别客户问题的解决方案。
*   **内容发现：** 推荐符合用户兴趣的文章、视频或其他内容。

通过结合关键词和语义匹配，混合搜索提供了更强大、更用户友好的搜索体验。

In [49]:
query = "What were the compute requirements for training GPT 4"

results = vector_search.similarity_search_with_score(
    query=query,
    k=5,
    query_type=CosmosDBQueryType.HYBRID,
)

# Display results
for i in range(0, len(results)):
    print(f"Result {i+1}: ", results[i][0].json())
    print(f"Score {i+1}: ", results[i][1])
    print("\n")

Result 1:  {"id":null,"metadata":{"source":"https://arxiv.org/pdf/2303.08774.pdf","page":97,"id":"36dfcd6c-d3cf-4e34-a5d6-cc4d63013cba"},"page_content":"Figure 11: Results on IF evaluations across GPT3.5, GPT3.5-Turbo, GPT-4-launch\n98","type":"Document"}
Score 1:  0.8173275975778744


Result 2:  {"id":null,"metadata":{"source":"https://arxiv.org/pdf/2303.08774.pdf","page":7,"id":"3d6e4715-4a38-40b1-89f1-e768bad5f9c8"},"page_content":"Preliminary results on a narrow set of academic vision benchmarks can be found in the GPT-4 blog\npost [ 65]. We plan to release more information about GPT-4’s visual capabilities in follow-up work.\n8","type":"Document"}
Score 2:  0.8176419674549597


Result 3:  {"id":null,"metadata":{"source":"https://arxiv.org/pdf/2303.08774.pdf","page":2,"id":"f2746fd3-bbcb-4197-b2d5-ee7b355b6009"},"page_content":"the HumanEval dataset. A power law fit to the smaller models (excluding GPT-4) is shown as the dotted\nline; this fit accurately predicts GPT-4’s performanc

## 混合搜索与过滤

Hybrid search combines the strengths of keyword search and vector search. Keyword search is good at matching exact terms, while vector search is good at finding semantically similar content. By combining them, you can get more relevant results.

This guide shows you how to implement hybrid search with filtering using the `azure-search-documents` library.

**Prerequisites**

*   An Azure AI Search service.
*   An index with at least one text field and one vector field.
*   The `azure-search-documents` library installed.

```bash
pip install azure-search-documents
```

**Steps**

1.  **Import necessary libraries:**

    ```python
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient
    from azure.search.documents.models import (
        QueryType,
        VectorFilterMode,
        VectorSearch,
        VectorQuery,
        KeywordValuePair,
        SearchMode,
    )
    ```

2.  **Set up the Search Client:**

    Replace `<YOUR_SEARCH_SERVICE_ENDPOINT>` and `<YOUR_SEARCH_API_KEY>` with your actual Azure AI Search service endpoint and API key.

    ```python
    endpoint = "<YOUR_SEARCH_SERVICE_ENDPOINT>"
    key = "<YOUR_SEARCH_API_KEY>"
    credential = AzureKeyCredential(key)
    search_client = SearchClient(endpoint, "your-index-name", credential)
    ```

3.  **Define your search query:**

    This example uses a simple keyword query and a vector query. You can also add filters to narrow down your search results.

    ```python
    keyword_query = "laptop"
    vector_query = VectorQuery(
        vector=...,  # Replace with your vector embedding
        k_nearest_neighbors=3,
        fields="content_vector"
    )

    # Example of filtering by a specific category
    filter_query = "category eq 'electronics'"
    ```

4.  **Perform the hybrid search:**

    Use `search_client.search` with `query_type="hybrid"` and `vector_queries=[vector_query]`. You can also specify `filter` for filtering.

    ```python
    results = search_client.search(
        search_text=keyword_query,
        vector_queries=[vector_query],
        filter=filter_query,
        query_type="hybrid",
        vector_filter_mode=VectorFilterMode.POST_FILTER, # or VectorFilterMode.PRE_FILTER
        search_mode=SearchMode.ALL, # or SearchMode.ANY
        top_k=5
    )

    for result in results:
        print(f"Score: {result['@search.score']}, Title: {result['title']}")
    ```

**Explanation of Parameters:**

*   `search_text`: The keyword query.
*   `vector_queries`: A list of `VectorQuery` objects.
*   `filter`: A filter expression to narrow down the search results.
*   `query_type`: Set to `"hybrid"` for hybrid search.
*   `vector_filter_mode`:
    *   `VectorFilterMode.POST_FILTER`: Filters are applied after the vector search. This is generally more flexible but can be slower.
    *   `VectorFilterMode.PRE_FILTER`: Filters are applied before the vector search. This can be faster but might exclude relevant results if the filter is too restrictive.
*   `search_mode`:
    *   `SearchMode.ALL`: Both keyword and vector search must match.
    *   `SearchMode.ANY`: Either keyword or vector search can match.
*   `top_k`: The number of results to return.

This example demonstrates a basic hybrid search with filtering. You can customize the queries, filters, and parameters to suit your specific needs.

In [51]:
query = "What were the compute requirements for training GPT 4"

pre_filter = {
    "conditions": [
        {
            "property": "text",
            "operator": "$full_text_contains_any",
            "value": "compute requirements",
        },
        {"property": "metadata.page", "operator": "$eq", "value": 0},
    ],
    "logical_operator": "$and",
}

results = vector_search.similarity_search_with_score(
    query=query, k=5, query_type=CosmosDBQueryType.HYBRID, pre_filter=pre_filter
)

# Display results
for i in range(0, len(results)):
    print(f"Result {i+1}: ", results[i][0].json())
    print(f"Score {i+1}: ", results[i][1])
    print("\n")

Result 1:  {"id":null,"metadata":{"source":"https://arxiv.org/pdf/2303.08774.pdf","page":97,"id":"36dfcd6c-d3cf-4e34-a5d6-cc4d63013cba"},"page_content":"Figure 11: Results on IF evaluations across GPT3.5, GPT3.5-Turbo, GPT-4-launch\n98","type":"Document"}
Score 1:  0.8173275975778744


Result 2:  {"id":null,"metadata":{"source":"https://arxiv.org/pdf/2303.08774.pdf","page":7,"id":"3d6e4715-4a38-40b1-89f1-e768bad5f9c8"},"page_content":"Preliminary results on a narrow set of academic vision benchmarks can be found in the GPT-4 blog\npost [ 65]. We plan to release more information about GPT-4’s visual capabilities in follow-up work.\n8","type":"Document"}
Score 2:  0.8176419674549597


Result 3:  {"id":null,"metadata":{"source":"https://arxiv.org/pdf/2303.08774.pdf","page":2,"id":"f2746fd3-bbcb-4197-b2d5-ee7b355b6009"},"page_content":"the HumanEval dataset. A power law fit to the smaller models (excluding GPT-4) is shown as the dotted\nline; this fit accurately predicts GPT-4’s performanc